# Full Misconception Extraction (train + test)

Runs the **chosen** prompt over the full train and test splits and saves the
per-turn misconception labels to `data/misconception/`. The modelling notebook
(04) reads these saved files.

- Set `CHOSEN_PROMPT` to the prompt you selected from notebook 02.
- Re-running overwrites the saved label files (rebuilt from the extraction
  cache, so the expensive API calls are reused; only uncached dialogues hit the
  API).

**Before running:** `export OPENROUTER_API_KEY=...` in the launching terminal,
or load it from a `.env` (see notebook 02).


## Setup

In [1]:
import os
from pathlib import Path
_here = Path.cwd()
for _c in [_here, *_here.parents]:
    if (_c / "data" / "annotated").exists():
        os.chdir(_c); break
import sys
sys.path.insert(0, str(Path.cwd() / "extension"))
print("cwd:", os.getcwd(), "| key set:", bool(os.environ.get("OPENROUTER_API_KEY")))


cwd: /Users/tandon.utsav2/Desktop/Experiment_1 | key set: True


In [2]:
from ast import literal_eval
import pandas as pd
from myext import filtering, prompt_loader, extraction, labels_io

CONV = {c: literal_eval for c in ["annotation", "dialogue"]}
def load(split):
    df = pd.read_csv(f"data/annotated/mathdial_{split}_atc.csv", converters=CONV)
    df, n = filtering.drop_failed_annotations(df)
    df.index = df["index"]
    print(f"{split}: {len(df)} dialogues ({n} failed-annotation removed)")
    return df

train = load("train")
test = load("test")


train: 2235 dialogues (18 failed-annotation removed)
test: 588 dialogues (7 failed-annotation removed)


## Choose the prompt

Set this to the prompt you selected on the validation set in notebook 02. The
full extraction uses this single prompt over every dialogue.


In [3]:
CHOSEN_PROMPT = "codebook_detailed"     # <-- set to your selected prompt
MODEL = "anthropic/claude-opus-4.8"

template = prompt_loader.load_prompt(CHOSEN_PROMPT)
print("using prompt:", CHOSEN_PROMPT, "| model:", MODEL)
print("available prompts:", prompt_loader.list_prompts())


using prompt: codebook_detailed | model: anthropic/claude-opus-4.8
available prompts: ['codebook_concise', 'codebook_concise_familyonly', 'codebook_detailed', 'codebook_detailed_familyonly', 'few_shot', 'few_shot_familyonly', 'minimal', 'minimal_familyonly', 'negative_guard', 'negative_guard_familyonly', 'opportunity_framing', 'opportunity_framing_familyonly', 'role_expert', 'role_expert_familyonly', 'stepwise_define', 'stepwise_define_familyonly']


## Extract over train and test

Per-dialogue calls, cached per (dialogue, prompt, model). The first full run is
the costly one; reruns are free from cache. Failures are counted and skipped,
not fatal.


In [4]:
train_extracted = extraction.extract_many(
    train, list(train.index), template, CHOSEN_PROMPT, model=MODEL)


  [codebook_detailed] 10/2235 done (0 errors)
  [codebook_detailed] 20/2235 done (0 errors)
  [codebook_detailed] 30/2235 done (0 errors)
  [codebook_detailed] 40/2235 done (0 errors)
  [codebook_detailed] 50/2235 done (0 errors)
  [codebook_detailed] 60/2235 done (0 errors)
  [codebook_detailed] 70/2235 done (0 errors)
  [codebook_detailed] 80/2235 done (0 errors)
  [codebook_detailed] 90/2235 done (0 errors)
  [codebook_detailed] 100/2235 done (0 errors)
  [codebook_detailed] 110/2235 done (0 errors)
  [codebook_detailed] 120/2235 done (0 errors)
  [codebook_detailed] 130/2235 done (0 errors)
  [codebook_detailed] 140/2235 done (0 errors)
  [codebook_detailed] 150/2235 done (0 errors)
  [codebook_detailed] 160/2235 done (0 errors)
  [codebook_detailed] 170/2235 done (0 errors)
  [codebook_detailed] 180/2235 done (0 errors)
  [codebook_detailed] 190/2235 done (0 errors)
  [codebook_detailed] 200/2235 done (0 errors)
  [codebook_detailed] 210/2235 done (0 errors)
  [codebook_detailed] 

In [5]:
test_extracted = extraction.extract_many(
    test, list(test.index), template, CHOSEN_PROMPT, model=MODEL)


  [codebook_detailed] 10/588 done (0 errors)
  [codebook_detailed] 20/588 done (0 errors)
  [codebook_detailed] 30/588 done (0 errors)
  [codebook_detailed] 40/588 done (0 errors)
  [codebook_detailed] 50/588 done (0 errors)
  [codebook_detailed] 60/588 done (0 errors)
  [codebook_detailed] 70/588 done (0 errors)
  [codebook_detailed] 80/588 done (0 errors)
  [codebook_detailed] 90/588 done (0 errors)
  [codebook_detailed] 100/588 done (0 errors)
  [codebook_detailed] 110/588 done (0 errors)
  [codebook_detailed] 120/588 done (0 errors)
  [codebook_detailed] 130/588 done (0 errors)
  [codebook_detailed] 140/588 done (0 errors)
  [codebook_detailed] 150/588 done (0 errors)
  [codebook_detailed] 160/588 done (0 errors)
  [codebook_detailed] 170/588 done (0 errors)
  [codebook_detailed] 180/588 done (0 errors)
  [codebook_detailed] 190/588 done (0 errors)
  [codebook_detailed] 200/588 done (0 errors)
  [codebook_detailed] 210/588 done (0 errors)
  [codebook_detailed] 220/588 done (0 error

## Save to data/misconception/ (overwrite)

Writes `train_labels.csv` and `test_labels.csv` with columns
`dialogue_id, turn, misc, family`. Overwrites any existing files, so this
notebook is the single source of the saved labels.


In [6]:
train_path = labels_io.save_labels("train", train_extracted, train)
test_path = labels_io.save_labels("test", test_extracted, test)

# quick sanity: label and family distributions
import pandas as pd
tl = pd.read_csv(train_path)
print("\ntrain misc distribution:", tl["misc"].value_counts(normalize=True).round(3).to_dict())
print("train family distribution:", tl["family"].value_counts().to_dict())


saved 13219 turn-labels for train -> data/misconception/train_labels.csv
saved 3498 turn-labels for test -> data/misconception/test_labels.csv

train misc distribution: {'present': 0.526, 'absent': 0.322, 'not_evidenced': 0.152}
train family distribution: {'A': 2442, 'D': 2405, 'B': 2173, 'F': 2160, 'E': 2126, 'C': 1913}


### Provenance note

The saved files record which prompt and model produced them only implicitly
(via this notebook). If you compare prompts at the modelling stage, save under
distinct folders or note the prompt in your lab log, since re-running this
notebook with a different `CHOSEN_PROMPT` overwrites the files in place.
